# 실험 009 — Strong 증강의 세 구조 앙상블

분할 seed는 42입니다. 세 모델 모두 strong 증강으로 학습된 가중치를 사용합니다.
MobileNetV3 Large seed 42는 008에서, EfficientNet-B0 seed 43은 007에서 재사용합니다.
ResNet18 Places365 seed 44만 새로 학습합니다.
최종 제출 CSV는 조건부 T0.3과 조건부 T0.3 TTA 두 개입니다.


## 실행 전 사전 준비

### 모듈
Colab은 필요한 모듈이 모두 설치되어 있습니다. 로컬이라면:
```shell
# 본인 환경(CPU/CUDA)에 맞는 PyTorch 설치: https://pytorch.org/get-started/locally/
pip3 install torch torchvision
pip3 install -U scikit-learn matplotlib
```

### 데이터 파일
운영진이 배포한 데이터를 노트북과 같은 폴더에 압축해제해 주세요 (Colab 은 `/content`).
```
📁root
 ├─📁train
 │  ├─📁6e      ← 6공학관 (label 0)
 │  └─📁it      ← IT관   (label 1)
 ├─📁test       ← 테스트 500장 (Public 250 + Private 250)
 └─📜baseline.ipynb
```


## 모듈 불러오기 & 시드 고정

재현성을 위해 모든 난수 시드를 고정합니다. 시드가 고정되어 있어야 "내가 바꾼 것 때문에 점수가 오른 것인지, 운이 좋았던 것인지"를 구분할 수 있습니다.

In [ ]:
import csv
import os
import random
import shutil
import subprocess
import time

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 42
seed_everything(SEED)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())

## 하이퍼 파라미터

In [ ]:
params = {
    'epoch': 50,
    'early_stopping': True,
    'min_epochs': 5,       # 각 학습률의 최소 학습 epoch, 이후부터 미개선 횟수 집계
    'patience': 5,
    'batch_size': 32,
    'learning_rates': (1e-3, 3e-4, 1e-4),
    'valid_ratio': 0.15,   # 학습 2,906장 중 검증으로 떼어둘 비율
    'workers': 0,          # Windows 노트북의 워커 프로세스 멈춤 방지; 모든 실험에 공통 적용
    'num_show_img': 5,     # 데이터 확인용 표시 개수 (학습과 무관)
}

In [ ]:
class_names = {0: "6e", 1: "it"}

RUN_SEEDS = (SEED, SEED + 2, SEED + 1)
MODEL_NAMES = ("mobilenet_v3_large", "resnet18_places365", "efficientnet_b0")
print("분할 시드:", SEED)
print("모델·학습 시드:", list(zip(MODEL_NAMES, RUN_SEEDS)))


## 결과 폴더와 제출 이름

실행 파일은 baseline_test3.ipynb입니다.
결과는 실험결과/009_Strong3종앙상블/seed_{SEED}/에 저장합니다.
submission_{SEED}_T0.3.csv와 submission_{SEED}_T0.3_TTA.csv를 생성합니다.
실행 전에 노트북을 저장합니다. 같은 seed로 재실행하면 009의 해당 결과를 덮어씁니다.


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
assert (BASE_DIR / "train").is_dir(), f"작업 폴더에 train이 없습니다: {BASE_DIR}"

RESULT_DIR = BASE_DIR / "실험결과" / "009_Strong3종앙상블" / f"seed_{SEED}"
GRAPH_DIR = RESULT_DIR / "graphs"
CHECKPOINT_DIR = RESULT_DIR / "checkpoints"
EPOCH_DIR = RESULT_DIR / "epochs"
CODE_DIR = RESULT_DIR / "코드"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EPOCH_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR.mkdir(parents=True, exist_ok=True)

print("그래프 저장 폴더:", GRAPH_DIR)
print("학습 가중치 저장 폴더:", CHECKPOINT_DIR)

print("epoch CSV 저장 폴더:", EPOCH_DIR)


MOBILENET_SOURCE_DIR = BASE_DIR / "실험결과" / "008_이종모델앙상블" / f"seed_{SEED}"
EFFICIENT_SOURCE_DIR = BASE_DIR / "실험결과" / "007_Strong기반앙상블" / f"seed_{SEED}"
PLACES_WEIGHT_PATH = BASE_DIR / "resnet18_places365.pth.tar"
assert PLACES_WEIGHT_PATH.is_file(), f"Places365 사전학습 가중치가 없습니다: {PLACES_WEIGHT_PATH}"
print("MobileNet 재사용:", MOBILENET_SOURCE_DIR)
print("EfficientNet 재사용:", EFFICIENT_SOURCE_DIR)


## 모델 정의 — model_test3.py 생성

MobileNetV3 Large·ResNet18 Places365·EfficientNet-B0를 사용합니다.
Places365 원본 가중치는 ResNet의 학습 초기화에 사용합니다.
최종 추론은 ZIP에 담긴 세 모델의 전체 가중치를 불러옵니다.


In [ ]:
%%writefile model_test3.py
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from torchvision.transforms import functional as F

IMG_SIZE = 256
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)
PAD_FILL = tuple(round(value * 255) for value in NORM_MEAN)
CONSENSUS_TEMPERATURE = 0.3
DISAGREEMENT_TEMPERATURE = 0.7


def resize_pad(image):
	width, height = image.size
	scale = IMG_SIZE / max(width, height)
	new_width = max(1, round(width * scale))
	new_height = max(1, round(height * scale))
	image = F.resize(image, [new_height, new_width], antialias=True)
	left = (IMG_SIZE - new_width) // 2
	top = (IMG_SIZE - new_height) // 2
	return F.pad(image, [left, top, IMG_SIZE - new_width - left, IMG_SIZE - new_height - top], fill=PAD_FILL)


class RandomAreaCrop:
	def __init__(self, scale):
		self.scale = scale

	def __call__(self, image):
		top, left, height, width = transforms.RandomResizedCrop.get_params(image, scale=self.scale, ratio=(0.75, 4 / 3))
		return F.crop(image, top, left, height, width)


def build_valid_transform():
	return transforms.Compose([
		transforms.Lambda(resize_pad),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])


def combine_probabilities(member_probs):
	assert (member_probs.ndim == 2) and (member_probs.shape[1] == 3)
	votes = member_probs >= 0.5
	unanimous = (votes == votes[:, :1]).all(dim=1, keepdim=True)
	median_probs = member_probs.sort(dim=1).values[:, 1:2]
	probs = torch.where(unanimous, member_probs.mean(dim=1, keepdim=True), median_probs)
	temperatures = torch.where(unanimous, CONSENSUS_TEMPERATURE, DISAGREEMENT_TEMPERATURE)
	return torch.sigmoid(torch.logit(probs) / temperatures)


def build_model(model_name, pretrained=False, places_weight_path=None):
	if model_name == "resnet18_places365":
		net = models.resnet18(weights=None, num_classes=365)
		if pretrained:
			if places_weight_path is None:
				raise ValueError("Places365 사전학습 가중치 경로가 필요합니다.")
			checkpoint = torch.load(places_weight_path, map_location="cpu", weights_only=True)
			state_dict = {key.removeprefix("module."): value for key, value in checkpoint["state_dict"].items()}
			net.load_state_dict(state_dict)
		net.fc = nn.Linear(net.fc.in_features, 1)
	elif model_name == "mobilenet_v3_large":
		weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
		net = models.mobilenet_v3_large(weights=weights)
		net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	elif model_name == "efficientnet_b0":
		weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
		net = models.efficientnet_b0(weights=weights)
		net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	else:
		raise ValueError(f"알 수 없는 모델: {model_name}")
	return net


class MixedEnsemble(nn.Module):
	def __init__(self, model_names, state_dicts):
		super().__init__()
		assert len(model_names) == len(state_dicts) == 3, "모델 이름과 가중치가 각각 3개 필요합니다."
		self.models = nn.ModuleList()
		for model_name, state_dict in zip(model_names, state_dicts):
			model = build_model(model_name)
			model.load_state_dict(state_dict)
			self.models.append(model)

	def member_probabilities(self, inputs, tta=False):
		# CPU 제출 재현과의 수치 차이를 줄이기 위해 추론에서는 TF32를 끕니다.
		with torch.backends.cudnn.flags(enabled=torch.backends.cudnn.enabled, benchmark=torch.backends.cudnn.benchmark, deterministic=torch.backends.cudnn.deterministic, allow_tf32=False):
			probs = torch.cat([torch.sigmoid(model(inputs)) for model in self.models], dim=1)
			if tta:
				flipped = inputs.flip(dims=[3])
				flipped_probs = torch.cat([torch.sigmoid(model(flipped)) for model in self.models], dim=1)
				probs = (probs + flipped_probs) / 2
		return probs

	def forward(self, inputs, tta=False):
		return combine_probabilities(self.member_probabilities(inputs, tta=tta))


In [ ]:
import importlib
import model_test3 as model_module

importlib.reload(model_module)


### 파라미터 제한

각 실행에서 모델을 새로 만들고 총 파라미터 수가 20,000,000 이하인지 확인합니다.


In [ ]:
PARAM_LIMIT = 20_000_000
print("MobileNetV3 Large + ResNet18 Places365 + EfficientNet-B0 / 예상 파라미터 19,389,167")


## 세 모델 공통 strong 증강

크롭 면적 50–100%, 밝기·대비 ±40%, 채도 ±30%, 좌우 반전 확률 0.5를 사용합니다.
가림과 원근 변형은 적용하지 않습니다.
검증·추론은 원본 비율을 유지해 256×256으로 중앙 패딩합니다.


In [ ]:
data_path = BASE_DIR / "train"
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD

AUGMENTATION_SETTINGS = {
	"strong": {"scale_min": 0.5, "brightness": 0.4, "contrast": 0.4, "saturation": 0.3, "erase_p": 0.0, "perspective_p": 0.0, "distortion_scale": 0.0},
}
train_transform = transforms.Compose([
	model_module.RandomAreaCrop(scale=(0.5, 1.0)),
	transforms.RandomHorizontalFlip(p=0.5),
	transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
	transforms.Lambda(model_module.resize_pad),
	transforms.ToTensor(),
	transforms.Normalize(NORM_MEAN, NORM_STD),
])
train_transforms = {"strong": train_transform}
valid_transform = model_module.build_valid_transform()
full_train = datasets.ImageFolder(data_path, train_transform)
full_valid = datasets.ImageFolder(data_path, valid_transform)
print("클래스 → 라벨 매핑:", full_train.class_to_idx)
print("전체 이미지 수:", len(full_train))
print("입력 크기:", IMG_SIZE, "비율 보존 및 중앙 패딩")


### 학습·검증 분할

분할 seed는 SEED이며 검증 비율은 15%입니다.
재사용할 두 모델의 원본 검증 ID·정답과 현재 분할이 같은지 확인합니다.


In [ ]:
labels = [y for _, y in full_train.samples]
train_idx, valid_idx = train_test_split(
    list(range(len(full_train))),
    test_size=params['valid_ratio'],
    stratify=labels,
    random_state=SEED,
)

data_sets = {
    'train': Subset(full_train, train_idx),
    'valid': Subset(full_valid, valid_idx),
}
print(f"train: {len(data_sets['train'])}장 / valid: {len(data_sets['valid'])}장")

In [ ]:
g = torch.Generator()
g.manual_seed(SEED)

dataloaders = {
    'train': DataLoader(data_sets['train'], batch_size=params['batch_size'],
                        shuffle=True, num_workers=params['workers'], generator=g),
    'valid': DataLoader(data_sets['valid'], batch_size=params['batch_size'],
                        shuffle=False, num_workers=params['workers']),
}
print('배치 수 — train: %d / valid: %d' % (len(dataloaders['train']), len(dataloaders['valid'])))

### 데이터 확인

In [ ]:
def img_show(tensor, title=None):
    arr = tensor.numpy().transpose((1, 2, 0))
    arr = arr * np.array(NORM_STD) + np.array(NORM_MEAN)  # 정규화 되돌리기 (표시용)
    arr = np.clip(arr, 0, 1)
    plt.figure(figsize=(12, 4))
    plt.imshow(arr)
    plt.axis('off')
    if title is not None:
        plt.title(title)
    plt.show()


inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs[:params['num_show_img']])
img_show(out, title=[class_names[int(c)] for c in classes[:params['num_show_img']]])

## 재사용 확인과 ResNet 학습

MobileNet seed SEED와 EfficientNet seed SEED+1은 같은 분할의 008·007에서 재사용합니다.
ResNet18 Places365는 seed SEED+2로 새로 학습합니다.
학습률·종료·선택 기준은 기존 실험과 같습니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, total_correct, total_n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

            logits = model(inputs)
            loss = criterion(logits, targets)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            total_loss += loss.item() * inputs.size(0)
            total_correct += (preds == targets).sum().item()
            total_n += inputs.size(0)

    return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs, epoch_csv, early_stopping, min_epochs, patience, learning_rates):
	assert min_epochs > 0
	assert patience > 0
	assert len(learning_rates) > 0
	assert all(rate > 0 for rate in learning_rates)
	since = time.time()
	history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
	best_acc, best_loss, best_epoch = -float('inf'), float('inf'), -1
	stage_index, stage_epochs = 0, 0
	stage_best_loss = float('inf')
	epochs_without_improvement = 0
	epoch_writer = csv.writer(epoch_csv)
	epoch_writer.writerow(('epoch', 'learning_rate', 'lr_stage', 'stage_epoch', 'train_loss', 'train_acc', 'valid_loss', 'valid_acc'))

	for epoch in range(1, num_epochs + 1):
		learning_rate = learning_rates[stage_index]
		for group in optimizer.param_groups:
			group['lr'] = learning_rate
		stage_epochs += 1
		tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
		va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

		history['train_loss'].append(tr_loss)
		history['train_acc'].append(tr_acc)
		history['valid_loss'].append(va_loss)
		history['valid_acc'].append(va_acc)
		epoch_writer.writerow((epoch, learning_rate, stage_index + 1, stage_epochs, tr_loss, tr_acc, va_loss, va_acc))
		epoch_csv.flush()
		print(f'Epoch {epoch}/{num_epochs} | lr {learning_rate:g} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | valid loss {va_loss:.4f}, acc {va_acc:.3f}')

		if (va_loss < best_loss) or ((va_loss == best_loss) and (va_acc > best_acc)):
			best_acc, best_loss, best_epoch = va_acc, va_loss, epoch
			torch.save(model.state_dict(), pt_file_name)
			print(f'  ↳ best Log Loss 갱신! {pt_file_name} 저장')

		if va_loss < stage_best_loss:
			stage_best_loss = va_loss
			epochs_without_improvement = 0
		elif stage_epochs > min_epochs:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			if stage_index + 1 < len(learning_rates):
				stage_index += 1
				stage_epochs = 0
				stage_best_loss = float('inf')
				epochs_without_improvement = 0
				print(f'다음 epoch부터 학습률 {learning_rates[stage_index]:g} 적용')
			elif early_stopping:
				print(f'조기 종료: epoch {epoch}, 마지막 학습률에서 Log Loss {patience}회 연속 미개선')
				break

	elapsed = time.time() - since
	print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
	print(f'best: epoch {best_epoch}, valid acc {best_acc:.4f}, log loss {best_loss:.4f}')
	return history, best_epoch, best_loss


In [ ]:
reused_results = []
source_specs = (
	(MOBILENET_SOURCE_DIR, "008", MODEL_NAMES[0], RUN_SEEDS[0]),
	(EFFICIENT_SOURCE_DIR, "007", MODEL_NAMES[2], RUN_SEEDS[2]),
)
expected_validation = {Path(full_valid.samples[i][0]).stem: int(full_valid.samples[i][1]) for i in valid_idx}
for source_dir, experiment, model_name, seed in source_specs:
	assert (source_dir / "runs.csv").is_file(), f"완료 기록이 없습니다: {source_dir}"
	with (source_dir / "runs.csv").open(newline="", encoding="utf-8-sig") as f:
		source_runs = list(csv.DictReader(f))
	matches = [row for row in source_runs if (row["model"] == model_name) and (int(row["seed"]) == seed)]
	assert len(matches) == 1, f"{model_name} seed {seed} 완료 행이 한 개 필요합니다."
	row = dict(matches[0])
	assert int(row["split_seed"]) == SEED
	assert row["augmentation"] == "strong", "순수 strong 모델만 재사용합니다."
	assert row["selection_metric"] == "valid_loss"
	assert row["preprocessing"] == "crop_then_aspect_resize_pad"
	assert int(row["image_size"]) == IMG_SIZE
	assert row["learning_rates"] == str(params["learning_rates"])
	assert int(row["max_epochs"]) == params["epoch"]
	assert int(row["min_epochs"]) == params["min_epochs"]
	assert int(row["patience"]) == params["patience"]
	assert row["early_stopping"] == str(params["early_stopping"])
	for key, value in AUGMENTATION_SETTINGS["strong"].items():
		assert float(row[key]) == value, f"증강 설정 불일치: {model_name}, {key}"
	with (source_dir / "validation_predictions.csv").open(newline="", encoding="utf-8-sig") as f:
		source_validation = list(csv.DictReader(f))
	assert len(source_validation) == len(expected_validation)
	assert {r["id"]: int(r["label"]) for r in source_validation} == expected_validation, "재사용 모델의 검증 분할이 다릅니다."
	for key in ("split_seed", "seed", "image_size", "max_epochs", "epochs_run", "min_epochs", "patience", "best_epoch", "n_params"):
		row[key] = int(row[key])
	for key in ("valid_loss", "valid_acc", "seconds", "valid_loss_first12", "valid_acc_first12"):
		row[key] = float(row[key])
	stem = f"{model_name}_seed{seed}"
	source_checkpoint = source_dir / "checkpoints" / f"{stem}.pt"
	assert source_checkpoint.is_file(), f"재사용 가중치가 없습니다: {source_checkpoint}"
	assert (source_dir / "epochs" / f"{stem}.csv").is_file()
	assert (source_dir / "graphs" / f"{stem}.png").is_file()
	row["source_experiment"] = experiment
	row["source_checkpoint"] = str(source_checkpoint)
	row["reused"] = True
	row["checkpoint"] = str(CHECKPOINT_DIR / f"{stem}.pt")
	reused_results.append(row)
for source_dir, filenames in (
	(MOBILENET_SOURCE_DIR, ("baseline_test2.ipynb", "model_test2.py")),
	(EFFICIENT_SOURCE_DIR, ("baseline_test.ipynb", "model.py")),
):
	for filename in filenames:
		assert (source_dir / "코드" / filename).is_file(), f"재사용 학습 코드가 없습니다: {filename}"
results = []
print("재사용 준비 완료:", [(row["model"], row["seed"]) for row in reused_results])


In [ ]:
assert RUN_SEEDS == (SEED, SEED + 2, SEED + 1), "모델 시드 설정 셀을 다시 실행하세요. ResNet seed는 SEED+2입니다."
# 실제 실행은 프로젝트 루트에서 수행하고, 저장된 학습 코드를 실험 폴더에 복사합니다.
for filename in ("baseline_test3.ipynb", "model_test3.py"):
	shutil.copy2(BASE_DIR / filename, CODE_DIR / filename)

for seed in RUN_SEEDS[1:2]:
	augmentation = "strong"
	settings = AUGMENTATION_SETTINGS[augmentation]
	full_run_train = datasets.ImageFolder(data_path, train_transforms[augmentation])
	assert full_run_train.samples == full_train.samples, "이미지 순서 또는 라벨이 달라졌습니다."
	run_data_sets = {"train": Subset(full_run_train, train_idx), "valid": data_sets["valid"]}
	print(f"\n{'=' * 60}")
	print(f"모델: resnet18_places365 / seed: {seed} / 증강: {augmentation}", flush=True)
	seed_everything(seed)

	model = model_module.build_model("resnet18_places365", pretrained=True, places_weight_path=PLACES_WEIGHT_PATH).to(device)
	n_params = sum(p.numel() for p in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 제한 초과: {n_params:,}"
	print(f"파라미터 수: {n_params:,}")

	# 모델 생성에서 소비한 난수와 데이터 로딩 난수를 분리합니다.
	seed_everything(seed)
	dataloaders = {
		"train": DataLoader(run_data_sets["train"], batch_size=params["batch_size"], shuffle=True, num_workers=params["workers"], generator=torch.Generator().manual_seed(seed)),
		"valid": DataLoader(run_data_sets["valid"], batch_size=params["batch_size"], shuffle=False, num_workers=params["workers"], generator=torch.Generator().manual_seed(seed)),
	}
	criterion = nn.BCEWithLogitsLoss()
	optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rates"][0])
	pt_file_name = CHECKPOINT_DIR / f"resnet18_places365_seed{seed}.pt"

	if device.type == "cuda":
		torch.cuda.synchronize(device)
	started = time.perf_counter()

	with (EPOCH_DIR / f"resnet18_places365_seed{seed}.csv").open("w", newline="", encoding="utf-8-sig") as epoch_csv:
		history, best_epoch, best_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=params["epoch"], epoch_csv=epoch_csv, early_stopping=params["early_stopping"], min_epochs=params["min_epochs"], patience=params["patience"], learning_rates=params["learning_rates"])

	if device.type == "cuda":
		torch.cuda.synchronize(device)
	elapsed = time.perf_counter() - started

	results.append({
		"model": "resnet18_places365",
		"source_experiment": "009",
		"source_checkpoint": "",
		"reused": False,
		"split_seed": SEED,
		"seed": seed,
		"augmentation": augmentation,
		"selection_metric": "valid_loss",
		"image_size": IMG_SIZE,
		"preprocessing": "crop_then_aspect_resize_pad",
		"erase_p": settings["erase_p"],
		"erase_scale": "(0.02, 0.1)",
		"perspective_p": settings["perspective_p"],
		"distortion_scale": settings["distortion_scale"],
		"scale_min": settings["scale_min"],
		"brightness": settings["brightness"],
		"contrast": settings["contrast"],
		"saturation": settings["saturation"],
		"learning_rates": str(params["learning_rates"]),
		"valid_loss_first12": min(history["valid_loss"][:12]),
		"valid_acc_first12": max(history["valid_acc"][:12]),
		"max_epochs": params["epoch"],
		"epochs_run": len(history["train_loss"]),
		"early_stopping": params["early_stopping"],
		"min_epochs": params["min_epochs"],
		"patience": params["patience"],
		"best_epoch": best_epoch,
		"valid_loss": best_loss,
		"valid_acc": history["valid_acc"][best_epoch - 1],
		"seconds": elapsed,
		"n_params": n_params,
		"checkpoint": str(pt_file_name),
	})

	# 실험 도중 중단돼도 완료된 실행의 지표는 CSV에 남깁니다.
	with (RESULT_DIR / "runs.csv").open("w", newline="", encoding="utf-8-sig") as f:
		writer = csv.DictWriter(f, fieldnames=list(results[0]))
		writer.writeheader()
		writer.writerows(results)

	epochs = range(1, len(history["train_loss"]) + 1)
	fig, axes = plt.subplots(1, 2, figsize=(12, 4))
	axes[0].plot(epochs, history["train_loss"], label="train")
	axes[0].plot(epochs, history["valid_loss"], label="valid")
	axes[0].plot(best_epoch, best_loss, "ro", label="best valid Log Loss")
	axes[0].set_ylabel("Log Loss")
	axes[1].plot(epochs, history["train_acc"], label="train")
	axes[1].plot(epochs, history["valid_acc"], label="valid")
	axes[1].plot(best_epoch, history["valid_acc"][best_epoch - 1], "ro", label="selected by valid Log Loss")
	axes[1].set_ylabel("Accuracy")
	for ax in axes:
		ax.set_xlabel("Epoch")
		ax.legend()
		ax.grid(alpha=0.3)
	fig.suptitle(f"resnet18_places365 / seed={seed} / {augmentation}")
	fig.tight_layout()
	fig.savefig(GRAPH_DIR / f"resnet18_places365_seed{seed}.png", dpi=160, bbox_inches="tight")
	plt.show()
	plt.close(fig)
	del optimizer, model, dataloaders


### 재사용 모델 복사와 구성 확정

ResNet 학습 후 두 원본 체크포인트·epoch CSV·그래프를 009 결과 폴더에 복사합니다.
구성 순서는 MobileNet, ResNet, EfficientNet입니다.


In [ ]:
resnet_results = [row for row in results if row["model"] == "resnet18_places365"]
assert len(resnet_results) == 1, "ResNet 학습 완료 결과가 한 개 필요합니다."
for row in reused_results:
	source_dir = Path(row["source_checkpoint"]).parent.parent
	stem = f"{row['model']}_seed{row['seed']}"
	shutil.copy2(row["source_checkpoint"], row["checkpoint"])
	shutil.copy2(source_dir / "epochs" / f"{stem}.csv", EPOCH_DIR / f"{stem}.csv")
	shutil.copy2(source_dir / "graphs" / f"{stem}.png", GRAPH_DIR / f"{stem}.png")
results = [reused_results[0], resnet_results[0], reused_results[1]]
with (RESULT_DIR / "runs.csv").open("w", newline="", encoding="utf-8-sig") as f:
	writer = csv.DictWriter(f, fieldnames=list(results[0]))
	writer.writeheader()
	writer.writerows(results)
print("앙상블 구성:", [(row["model"], row["seed"], row["reused"]) for row in results])


### 구성 모델 기록

모델 이름과 seed의 조합으로 구분합니다. ResNet과 EfficientNet의 seed가 동일해도 별도 모델로 기록합니다.
summary.csv는 세 구성 모델의 기술 통계이며 반복 학습의 성능 오차범위를 나타내지 않습니다.


In [ ]:
assert len(results) == len(RUN_SEEDS), "모든 시드의 실행을 완료해야 합니다."

losses = np.array([row["valid_loss"] for row in results])
accuracies = np.array([row["valid_acc"] for row in results])
seconds = np.array([row["seconds"] for row in results])
summary = {
	"model": "mobilenet_v3_large+resnet18_places365+efficientnet_b0",
	"split_seed": SEED,
	"runs": len(results),
	"loss_mean": float(losses.mean()),
	"loss_std": float(losses.std(ddof=1)),
	"accuracy_mean": float(accuracies.mean()),
	"accuracy_std": float(accuracies.std(ddof=1)),
	"seconds_mean": float(seconds.mean()),
	"seconds_std": float(seconds.std(ddof=1)),
	"n_params": sum(row["n_params"] for row in results),
}
with (RESULT_DIR / "summary.csv").open("w", newline="", encoding="utf-8-sig") as f:
	writer = csv.DictWriter(f, fieldnames=list(summary))
	writer.writeheader()
	writer.writerow(summary)

print(f"검증 Log Loss: {summary['loss_mean']:.4f} ± {summary['loss_std']:.4f}")
print(f"검증 정확도: {summary['accuracy_mean']:.2%} ± {summary['accuracy_std']:.2%}")
print(f"이번 ResNet 학습 시간: {results[1]['seconds'] / 60:.1f}분")
print("실행별 결과:", RESULT_DIR / "runs.csv")
print("전체 통계:", RESULT_DIR / "summary.csv")



### 세 모델의 조건부 T0.3과 TTA 검증

3:0 일치 시 세 확률 평균에 T=0.3, 2:1 불일치 시 가운데 확률에 T=0.7을 적용합니다.
TTA는 모델별 원본·좌우 반전 확률 평균 후 같은 결합 규칙을 적용합니다.


In [ ]:
assert [row["seed"] for row in results] == list(RUN_SEEDS)
assert [row["model"] for row in results] == list(MODEL_NAMES)
ensemble_weights_path = RESULT_DIR / "ensemble.pt"
checkpoint = {
	"model_names": list(MODEL_NAMES),
	"seeds": list(RUN_SEEDS),
	"split_seed": SEED,
	"augmentations": ["strong"] * 3,
	"image_size": IMG_SIZE,
	"preprocessing": "aspect_resize_pad",
	"temperatures": [model_module.CONSENSUS_TEMPERATURE, model_module.DISAGREEMENT_TEMPERATURE],
	"source_experiments": [row["source_experiment"] for row in results],
	"state_dicts": [torch.load(row["checkpoint"], map_location="cpu", weights_only=True) for row in results],
}
ensemble = model_module.MixedEnsemble(checkpoint["model_names"], checkpoint["state_dicts"]).to(device).eval()
ensemble_params = sum(p.numel() for p in ensemble.parameters())
assert ensemble_params <= PARAM_LIMIT, f"앙상블 파라미터 제한 초과: {ensemble_params:,}"
torch.save(checkpoint, ensemble_weights_path)
valid_loader = DataLoader(data_sets["valid"], batch_size=params["batch_size"], shuffle=False, num_workers=0)
validation_probabilities, validation_tta_probabilities, validation_targets = [], [], []
with torch.inference_mode():
	for inputs, targets in valid_loader:
		inputs = inputs.to(device)
		member_probs = ensemble.member_probabilities(inputs).cpu()
		flipped_probs = ensemble.member_probabilities(inputs.flip(dims=[3])).cpu()
		validation_probabilities.append(member_probs)
		validation_tta_probabilities.append((member_probs + flipped_probs) / 2)
		validation_targets.append(targets)
member_probs = torch.cat(validation_probabilities)
tta_member_probs = torch.cat(validation_tta_probabilities)
targets = torch.cat(validation_targets).float()
ensemble_probs_by_rule = {
	"T0.3": model_module.combine_probabilities(member_probs),
	"T0.3_TTA": model_module.combine_probabilities(tta_member_probs),
}
mean_probs = member_probs.mean(dim=1, keepdim=True)
median_probs = member_probs.sort(dim=1).values[:, 1:2]
all_probs = torch.cat((member_probs, mean_probs, median_probs, *ensemble_probs_by_rule.values()), dim=1)
names = list(MODEL_NAMES) + ["mean_T1", "median_T1"] + list(ensemble_probs_by_rule)
metrics = []
for index, name in enumerate(names):
	probs = all_probs[:, index]
	row = {
		"model": name, "samples": len(targets),
		"valid_acc": ((probs >= 0.5) == targets.bool()).float().mean().item(),
		"valid_loss": torch.nn.functional.binary_cross_entropy(probs.clamp(1e-6, 1 - 1e-6), targets).item(),
	}
	metrics.append(row)
	print(f"{name}: ACC {row['valid_acc']:.2%} / Log Loss {row['valid_loss']:.5f}")
for filename, rows in (("ensemble_metrics.csv", metrics), ("comparison.csv", metrics[-4:])):
	with (RESULT_DIR / filename).open("w", newline="", encoding="utf-8-sig") as f:
		writer = csv.DictWriter(f, fieldnames=list(rows[0]))
		writer.writeheader()
		writer.writerows(rows)
agreement_metrics = []
votes_by_rule = {}
for rule, probabilities in (("T0.3", member_probs), ("T0.3_TTA", tta_member_probs)):
	votes = (probabilities >= 0.5).sum(dim=1)
	votes_by_rule[rule] = votes
	for name, mask in (("3:0", (votes == 0) | (votes == 3)), ("2:1", (votes == 1) | (votes == 2))):
		count = int(mask.sum())
		row = {"rule": rule, "agreement": name, "samples": count, "valid_acc": None, "valid_loss": None}
		if count > 0:
			probs = ensemble_probs_by_rule[rule].squeeze(1)[mask]
			labels = targets[mask]
			row["valid_acc"] = ((probs >= 0.5) == labels.bool()).float().mean().item()
			row["valid_loss"] = torch.nn.functional.binary_cross_entropy(probs.clamp(1e-6, 1 - 1e-6), labels).item()
		agreement_metrics.append(row)
with (RESULT_DIR / "agreement_metrics.csv").open("w", newline="", encoding="utf-8-sig") as f:
	writer = csv.DictWriter(f, fieldnames=list(agreement_metrics[0]))
	writer.writeheader()
	writer.writerows(agreement_metrics)
with (RESULT_DIR / "validation_predictions.csv").open("w", newline="", encoding="utf-8-sig") as f:
	writer = csv.writer(f)
	header = ["id", "label", "positive_votes", "TTA_positive_votes", "T0.3_temperature", "TTA_temperature"]
	writer.writerow(header + [f"{name}_prob" for name in names] + [f"{name}_TTA_prob" for name in MODEL_NAMES])
	for position, sample_index in enumerate(valid_idx):
		temperatures = []
		for rule in ensemble_probs_by_rule:
			votes = int(votes_by_rule[rule][position])
			temperatures.append(model_module.CONSENSUS_TEMPERATURE if votes in (0, 3) else model_module.DISAGREEMENT_TEMPERATURE)
		image_id = Path(full_valid.samples[sample_index][0]).stem
		writer.writerow([image_id, int(targets[position]), int(votes_by_rule["T0.3"][position]), int(votes_by_rule["T0.3_TTA"][position])] + temperatures + all_probs[position].tolist() + tta_member_probs[position].tolist())
print(f"앙상블 파라미터: {ensemble_params:,}")
print("앙상블 가중치:", ensemble_weights_path)


## 테스트 추론 — 제출 CSV 두 개 생성

같은 가중치에서 조건부 T0.3과 조건부 T0.3 TTA를 생성합니다.


In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, files, transform):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        file = self.files[index]
        img = Image.open(file).convert('RGB')
        return self.transform(img), os.path.basename(file)


import glob
test_files = sorted(glob.glob('./test/*'))
print('테스트 이미지 수:', len(test_files))

test_loader = DataLoader(
    TestImageDataset(test_files, valid_transform),
    batch_size=params['batch_size'], shuffle=False)

In [ ]:
submission_rows = {name: ["id,prob"] for name in ("T0.3", "T0.3_TTA")}
ensemble.eval()
with torch.inference_mode():
	for inputs, filenames in test_loader:
		inputs = inputs.to(device)
		member_probs = ensemble.member_probabilities(inputs).cpu()
		flipped_probs = ensemble.member_probabilities(inputs.flip(dims=[3])).cpu()
		probabilities = {
			"T0.3": model_module.combine_probabilities(member_probs).squeeze(1),
			"T0.3_TTA": model_module.combine_probabilities((member_probs + flipped_probs) / 2).squeeze(1),
		}
		for name, probs in probabilities.items():
			for filename, prob in zip(filenames, probs):
				submission_rows[name].append(f"{Path(filename).stem},{prob.item():.6f}")
with (BASE_DIR / "sample_submission.csv").open(newline="", encoding="utf-8-sig") as f:
	expected_ids = [row["id"] for row in csv.DictReader(f)]
assert len(expected_ids) == len(set(expected_ids)) == 500
submission_paths = {}
for name, rows in submission_rows.items():
	image_ids = [row.split(",")[0] for row in rows[1:]]
	assert len(image_ids) == len(set(image_ids)) == 500
	assert set(image_ids) == set(expected_ids), "테스트 ID가 sample_submission과 다릅니다."
	assert all(np.isfinite(float(row.split(",")[1])) and (0 <= float(row.split(",")[1]) <= 1) for row in rows[1:]), "비정상 확률이 있습니다."
	path = RESULT_DIR / f"submission_{SEED}_{name}.csv"
	path.write_text("\n".join(rows) + "\n", encoding="utf-8")
	submission_paths[name] = path
	print("제출 CSV:", path)


## 제출 패키지

baseline_test3.ipynb·model_test3.py·predict_test3.py·model.pt·runs.csv를 포함합니다.
source_008/과 source_007/에 재사용한 두 모델의 학습 코드도 포함합니다.

```shell
python predict_test3.py --input test --output pred.csv
python predict_test3.py --input test --output pred_tta.csv --tta
```


In [ ]:
%%writefile predict_test3.py
import argparse
from pathlib import Path

import torch
from PIL import Image
from model_test3 import IMG_SIZE, MixedEnsemble, build_valid_transform

PARAM_LIMIT = 20_000_000


def main():
	parser = argparse.ArgumentParser()
	parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
	parser.add_argument("--output", required=True, help="출력 CSV 경로")
	parser.add_argument("--weights", default="model.pt", help="앙상블 가중치 파일")
	parser.add_argument("--tta", action="store_true", help="원본·좌우 반전 확률 평균 후 조건부 T0.3 적용")
	args = parser.parse_args()
	weights_path = Path(__file__).resolve().parent / args.weights
	checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
	assert checkpoint["image_size"] == IMG_SIZE, "입력 크기가 다릅니다."
	model = MixedEnsemble(checkpoint["model_names"], checkpoint["state_dicts"]).eval()
	n_params = sum(p.numel() for p in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 20M 초과: {n_params:,}"
	transform = build_valid_transform()
	files = sorted(p for p in Path(args.input).iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
	assert len(files) > 0, "테스트 이미지가 없습니다."
	rows = ["id,prob"]
	with torch.inference_mode():
		for start in range(0, len(files), 32):
			batch_files = files[start:start + 32]
			images = []
			for file in batch_files:
				with Image.open(file) as image:
					images.append(transform(image.convert("RGB")))
			probs = model(torch.stack(images), tta=args.tta).squeeze(1)
			for file, prob in zip(batch_files, probs):
				rows.append(f"{file.stem},{prob.item():.6f}")
	Path(args.output).write_text("\n".join(rows) + "\n", encoding="utf-8")
	print(f"{args.output} 저장 완료 ({len(files)}장, 파라미터 {n_params:,})")


if __name__ == "__main__":
	main()


In [ ]:
package_dir = RESULT_DIR / "package"
assert package_dir.resolve().parent == RESULT_DIR.resolve()
package_dir.mkdir(exist_ok=True)
shutil.copy2(BASE_DIR / "predict_test3.py", CODE_DIR / "predict_test3.py")
for filename in ("model_test3.py", "predict_test3.py", "baseline_test3.ipynb"):
	shutil.copy2(CODE_DIR / filename, package_dir / filename)
for folder, source_dir, filenames in (
	("source_008", MOBILENET_SOURCE_DIR, ("baseline_test2.ipynb", "model_test2.py")),
	("source_007", EFFICIENT_SOURCE_DIR, ("baseline_test.ipynb", "model.py")),
):
	source_code_dir = package_dir / folder
	source_code_dir.mkdir(exist_ok=True)
	for filename in filenames:
		shutil.copy2(source_dir / "코드" / filename, source_code_dir / filename)
shutil.copy2(RESULT_DIR / "runs.csv", package_dir / "runs.csv")
shutil.copyfile(ensemble_weights_path, package_dir / "model.pt")
zip_path = shutil.make_archive(str(RESULT_DIR / "submission_package"), "zip", package_dir)
assert Path(zip_path).stat().st_size <= 200_000_000, "ZIP 200MB 초과"
print("앙상블 패키지 생성:", zip_path)


### 패키지 CPU 자가 검증

패키지의 추론 코드로 두 CSV를 각각 재현하고 최대 확률 차이를 확인합니다.
Colab CPU의 500장 600초 제한은 실제 Colab에서도 확인합니다.


In [ ]:
import sys
subprocess_env = os.environ.copy()
subprocess_env["PYTHONIOENCODING"] = "utf-8"
for name, use_tta in (("T0.3", False), ("T0.3_TTA", True)):
	package_check_path = RESULT_DIR / f"package_check_{SEED}_{name}.csv"
	command = [sys.executable, str(package_dir / "predict_test3.py"), "--input", str(BASE_DIR / "test"), "--output", str(package_check_path)]
	if use_tta:
		command.append("--tta")
	started = time.perf_counter()
	result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", check=True, env=subprocess_env)
	elapsed = time.perf_counter() - started
	print(result.stdout)
	print(result.stderr)
	print(f"{name} CPU 추론 시간: {elapsed:.1f}초")
	assert elapsed <= 600, "CPU 추론 10분 초과"
	with submission_paths[name].open(newline="", encoding="utf-8") as f:
		nb_probs = {row["id"]: float(row["prob"]) for row in csv.DictReader(f)}
	with package_check_path.open(newline="", encoding="utf-8") as f:
		pkg_probs = {row["id"]: float(row["prob"]) for row in csv.DictReader(f)}
	assert nb_probs.keys() == pkg_probs.keys(), "파일 목록이 다릅니다."
	max_diff = max(abs(nb_probs[key] - pkg_probs[key]) for key in nb_probs)
	print(f"{name} 노트북 vs 패키지 확률 최대 차이: {max_diff:.2e}")
	assert max_diff < 1e-3, "전처리·가중치·앙상블·CPU/GPU 수치 차이를 확인하세요."
print("조건부 T0.3과 TTA 패키지 자가 검증 통과")


## 실행 확인

- SEED=42에서 MobileNet 42, ResNet 44, EfficientNet 43을 사용합니다.
- MobileNet과 EfficientNet은 원본 완료 기록·strong 설정·검증 분할을 확인하고 재사용합니다.
- 추가 학습은 ResNet18 Places365 한 개입니다.
- 두 제출 CSV의 ID·확률, 패키지 CPU 재현과 추론 시간을 확인합니다.
- SEED=1로 변경하면 모델 seed는 1·3·2가 됩니다. 같은 분할의 008 MobileNet과 007 EfficientNet 결과가 필요합니다.
